# `ClassBase`: learning an unfamiliar Nematics3D object

`nematics3d.core.class_base.ClassBase` is a low-level base class shared by many objects in Nematics3D. Users normally do **not** create a `ClassBase` directly. Instead, concrete classes inherit from it and therefore share a common way to describe themselves, organize attributes, control assignment, expose actions, and record relations to other objects.

The important consequence is practical: once you learn the small common vocabulary supplied by `ClassBase`, you do not need to memorize the full API of every Nematics3D object before you can start using it.

This tutorial starts from the most realistic situation: **you have just received an object and have almost no idea what it is or how to use it.**


## Setup: an object we have never used before

We will use `SmoothedLine` as the example. Its scientific role is simple enough for this tutorial: it stores a polyline and constructs a smoothed version of it.

`SmoothedLine` inherits from `HostBase`, which in turn inherits from `ClassBase`. We will ignore the additional `HostBase` machinery here. Everything introduced below is part of the common object language that a `ClassBase` descendant can expose.


In [ ]:
import numpy as np
import nematics3d as n3d

t = np.linspace(0.0, 4.0 * np.pi, 101)
coords = np.column_stack([
    t,
    np.sin(t) + 0.08 * np.sin(9.0 * t),
    0.25 * np.cos(0.5 * t),
])

line = n3d.SmoothedLine(
    coords,
    name="example line",
    window_length=11,
)

line


## I have this object. What do I do with it?

Suppose `line` was handed to you by somebody else. You may not know what class it belongs to, what data it contains, which fields are important, or what you are allowed to change.

A good first move in an editor, IPython, or Jupyter notebook is simply to type:

```python
line.show_
```

and invoke autocomplete (for example, press **Tab**).

The `show_` prefix groups the object's inspection and explanation methods. In other words, when you do not yet know an object, **ask the object to explain itself**.

Some of the most useful entries are:

| Method | Question it answers |
| --- | --- |
| `show_doc()` | What are you? What are you for? |
| `show_readable_attrs()` | What information do you contain? |
| `show_attr_doc(...)` | What does this particular attribute mean? |
| `show_attr_info(...)` | Give me the detailed information about this attribute. |
| `show_relations()` | What other objects are you related to? |
| `show_relation_tree()` | How are those object relations organized? |


## First question: what are you?

Start with the most basic question:


In [ ]:
line.show_doc()


`show_doc()` displays the class docstring of the **concrete class of this object**. Here it describes `SmoothedLine`, not `ClassBase`.

This is the quickest way to learn the object's overall role before looking at individual fields.


## Every object has a name

Before looking inside the object, notice the small piece of identity that every `ClassBase` descendant carries:


In [ ]:
line.name, line.raw_name


`name` is the convenient public form; `raw_name` is the canonical stored field underneath it. You will usually use `name` in everyday code. It also appears in representations, logs, and relation displays, making individual objects easier to recognize.

If the name should change, use the object's naming action:


In [ ]:
line.act_set_name("smoothed helix")
line


`act_set_name(...)` validates the new name before storing it. This becomes especially useful when objects enter a `RegistryBase`: the registry uses each object's `name` for lookup and keeps names unique. For now, the key idea is simply that every `ClassBase` object arrives with a readable identity.


## Second question: what do you contain?

Once you know the object's general purpose, ask what user-readable information it exposes:


In [ ]:
line.show_readable_attrs()


The result is more useful than a raw dump of Python internals: it presents the attributes that belong to the object's public vocabulary together with their descriptions.

If one name catches your attention, inspect only that field. For example:


In [ ]:
line.show_attr_doc("raw_coords")


If you want more than the documentation string, use:


In [ ]:
line.show_attr_info("raw_coords")


`show_attr_info(...)` combines information such as the canonical name, attribute kind, public alias when one exists, whether the field is modifiable or protected, its current value, and its documentation.

At this point, without reading the implementation, we already have a workable path through an unfamiliar object:

```text
line.show_  + autocomplete
        ↓
line.show_doc()
        ↓
line.show_readable_attrs()
        ↓
line.show_attr_doc("...")
line.show_attr_info("...")
```


## Why does this discovery workflow work so well? Prefixes are part of the API

The fact that typing `show_` immediately finds the explanation tools is not accidental. `ClassBase` uses explicit naming conventions so that names carry semantic information.

There are two closely related conventions:

- **method prefixes** tell you what kind of operation a method performs;
- **attribute prefixes** tell you what role a piece of data plays in the object.

This means autocomplete is not merely a convenience. It is one of the intended ways to discover a Nematics3D object's interface.


### Method prefixes: `show_` and `act_`

Public object methods commonly use two semantic prefixes:

| Prefix | Meaning | Typical question |
| --- | --- | --- |
| `show_...` | Inspect, display, or explain something | What can you tell me? |
| `act_...` | Perform an action on or through the object | What can you do? |

You have already used the first one:

```python
line.show_      # autocomplete: what can this object show or explain?
```

The same idea works for actions:

```python
line.act_       # autocomplete: what actions can this object perform?
```

Some `act_` methods come from more specialized subclasses such as `HostBase`. You do not need to understand them yet. The important point here is that their names already tell you that they **do something**, rather than merely report information.


### Attribute prefixes: what role does this value play?

Most structured `ClassBase` attributes belong to a semantic category indicated by their name:

| Prefix | Meaning | A useful way to read it |
| --- | --- | --- |
| `raw_...` | Canonical stored public input or base data | What was given to the object? |
| `state_...` | Writable runtime state | What state is the object currently in? |
| `default_...` | Managed default-layer input | What default value/settings does it carry? |
| `calc_...` | Computed readable data, normally read-only | What has the object calculated? |
| `entity_...` | Computed or generated object-valued result, normally read-only | What object has it created? |
| `impl_...` | Internal implementation state | Usually not part of the user-facing API |

There are also deliberately unprefixed public forms, notably semantic relations such as `owner` or `registry`, ordinary Python properties defined by a concrete class, and user-added extra attributes. Their meaning is documented separately rather than inferred from one of the prefixes above.


## Use the prefixes to explore instead of memorizing names

Now suppose you know that `line` has calculated several quantities, but you do not remember their names. Type:

```python
line.calc_
```

and invoke autocomplete. For `SmoothedLine`, this narrows the candidates to calculated fields such as `calc_coords`, `calc_result`, and `calc_status`.

The same pattern answers several common questions:

```python
line.raw_       # What base input does this object store?
line.state_     # What runtime state does it expose?
line.default_   # What default-layer values does it expose?
line.calc_      # What has it calculated?
line.entity_    # What object-valued results has it generated?
line.show_      # How can it explain itself?
line.act_       # What can it do?
```

A useful rule of thumb is therefore:

> **Do not start by memorizing every concrete class. Learn the `ClassBase` vocabulary, then let autocomplete show you which words this particular object provides.**


## A closer look at `raw_`: canonical input and its shorter alias

For `SmoothedLine`, the clearest `raw_` field is:


In [ ]:
line.raw_coords


This is the canonical stored input polyline. `raw_` fields normally also have a shorter readable alias with the prefix removed:


In [ ]:
line.raw_coords, line.coords


Both names refer to the same underlying input. `coords` is convenient in ordinary analysis code; `raw_coords` makes the semantic role explicit.

Compare that with:

```python
line.calc_result
```

The names already tell you that `raw_coords` is an input, while `calc_result` is produced by the object's calculation.

**Do not treat `raw_` fields as arbitrary mutable storage.** They are core inputs. Changing one can require derived `calc_` and `entity_` state to be rebuilt. The exact update behavior belongs to the concrete subclass (and, for host objects, to `HostBase`), so inspect the class before changing unfamiliar core inputs.


## Adding your own side information

Sometimes you want to attach information that is useful to your analysis but is **not** part of the class's scientific definition: a note, sample identifier, provenance label, or temporary annotation.

Use an extra attribute:


In [ ]:
line.act_add_attr(
    "note",
    "A user note attached to this particular line.",
    default="candidate smoothing result",
)

line.note


The new field joins the object's inspection interface:


In [ ]:
line.show_attr_info("note")


When it is no longer needed:


In [ ]:
line.act_remove_attr("note")


Extra attributes are side storage. They do not become `raw_`, `state_`, `calc_`, `entity_`, or `impl_` fields, and those semantic prefixes are reserved rather than available for user-added extras.


## Relations: how is this object connected to other objects?

A Nematics3D object may carry semantic links to other objects, for example an `owner` or a `registry`. To inspect the relations currently bound to an object, use:


In [ ]:
line.show_relations()


For a larger connected structure, use:

```python
line.show_relation_tree()
```

Relations describe semantic object connections. They should not be interpreted as a promise that changing one object automatically recomputes every related object. Any synchronization or dependency behavior is defined separately by the relevant concrete classes.


## A practical discovery workflow

When you encounter an unfamiliar Nematics3D object, you can usually begin with the following sequence:

```python
obj.show_                     # autocomplete: how can the object explain itself?
obj.show_doc()                # what is it?
obj.show_readable_attrs()     # what does it contain?
obj.show_attr_info("...")   # what exactly is this field?

obj.raw_                      # autocomplete: base inputs
obj.state_                    # autocomplete: runtime state
obj.calc_                     # autocomplete: calculated data
obj.entity_                   # autocomplete: generated objects
obj.act_                      # autocomplete: available actions
```

If the object participates in a larger object structure, also try:

```python
obj.show_relations()
obj.show_relation_tree()
```


`ClassBase` is not something most users need to instantiate. It is the common object protocol underneath many Nematics3D classes.

The most useful habit is therefore not to memorize every class-specific attribute. Instead:

1. **Read `name` as the object's identity.** It helps you recognize the object in code, logs, and larger object structures.
2. **Start with `show_` and autocomplete.** Let the object tell you what it is and what it contains.
3. **Read prefixes as semantic information.** `raw_`, `state_`, `default_`, `calc_`, `entity_`, and `impl_` tell you what role an attribute plays.
4. **Use autocomplete as API discovery.** `calc_`, `entity_`, `show_`, and `act_` immediately narrow a large object to the category you care about.

Once this vocabulary becomes familiar, a new Nematics3D object is no longer a completely new API. You already know how to start asking it the right questions.

A natural next step is `RegistryBase`, which builds directly on object names to store, find, and distinguish multiple `ClassBase` objects. `HostBase` extends the same object language in another direction, adding `opts`, commits, and dependent-state updates when inputs change.
